In [1]:
"""
PIPELINE STAGE: Data Merging, Spatial Mapping & International Standardization
PERIOD: January 2020 - October 2021
LIBRARIES: os, pandas, numpy, re

1. OBJECTIVE:
   Clean, standardize, and merge processed subscriber and consumption datasets into a final, 
   research-ready format. This script focuses on categorical mapping, geographic identification 
   (Plate codes), and English localization to meet international academic presentation standards 
   (e.g., PeerJ).

2. FILE ARCHITECTURE:
   - Input 1 (Subscriber): os.path.join("..", "..", "processed_data", "steps", "01_2020_2021_subscriber.xlsx")
   - Input 2 (Consumption): os.path.join("..", "..", "processed_data", "steps", "02_2020_2021_consumption.xlsx")
   - Output (Final): os.path.join("..", "..", "processed_data", "steps", "03_2020_2021_final.xlsx")

3. DATA CLEANING & REFINEMENT:
   A. Row Filtering & Merging: Remove redundant header rows (e.g., "İl Adı"). Perform a Left Join 
      to integrate 'Consumption' data into the subscriber dataframe using [Province, Year, Month] keys.
   B. Spatial Normalization & Plate Mapping: Standardize province nomenclature (e.g., "K.MARAŞ" -> 
      "KAHRAMANMARAŞ", "HAKKÂRİ" -> "HAKKARİ"). Map standardized province names to Turkey's official 
      numeric Plate Codes (1-81) to establish a robust geographic index.
   C. Temporal Transformation: Map English string month names to numerical integers (1-12).

4. NUMERICAL INTEGRITY & TRANSLATION (English Localization):
   - Target Columns: Lighting, Residential, Industrial, Agricultural, Public, Consumption.
   - Transformation: Remove thousand separators, strip decimal components, and cast all values 
     as integers. Convert null or erroneous entries to 0.
   - Translation: Translate Turkish headers to English (e.g., il -> Province, Ticarethane -> Public).
   - Pruning: Explicitly remove the 'Total_Subscribers' ('Toplam_Abone') column as per analytical requirements.

5. STRUCTURAL ORGANIZATION & LOGGING:
   - Sorting Logic: Order strictly by 'Plate' (Ascending), then 'Year' (Ascending), and 'Month' (Ascending).
   - Fixed Schema: [Plate, Province, Year, Month, Consumption, Lighting, Public, Residential, 
     Industrial, Agricultural]
   - Error Handling: Encapsulate workflow in a try-except block with real-time console logging.
"""


import pandas as pd
import os

# --- AYARLAR ---
file_path = os.path.join("..", "..","processed_data", "steps", "01_2020_2021_subscriber.xlsx")
consumption_path = os.path.join("..","..", "processed_data", "steps", "02_2020_2021_consumption.xlsx")
output_path = os.path.join("..","..","processed_data", "steps", "03_2020_2021_final.xlsx")

# Temizlenecek sayısal sütunlar (Toplam_Abone çıkarıldı)
numeric_columns = ['Aydinlatma', 'Mesken', 'Sanayi', 'Tarim', 'Ticarethane', 'Consumption']

# Ay isimlerini numaraya çevirme sözlüğü
MONTH_MAP = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12
}

# Plaka kodları sözlüğü
plaka_kodlari = {
    "ADANA": 1, "ADIYAMAN": 2, "AFYONKARAHİSAR": 3, "AĞRI": 4, "AMASYA": 5, "ANKARA": 6, "ANTALYA": 7, "ARTVİN": 8,
    "AYDIN": 9, "BALIKESİR": 10, "BİLECİK": 11, "BİNGÖL": 12, "BİTLİS": 13, "BOLU": 14, "BURDUR": 15, "BURSA": 16,
    "ÇANAKKALE": 17, "ÇANKIRI": 18, "ÇORUM": 19, "DENİZLİ": 20, "DİYARBAKIR": 21, "EDİRNE": 22, "ELAZIĞ": 23, "ERZİNCAN": 24,
    "ERZURUM": 25, "ESKİŞEHİR": 26, "GAZİANTEP": 27, "GİRESUN": 28, "GÜMÜŞHANE": 29, "HAKKARİ": 30, "HATAY": 31, "ISPARTA": 32,
    "MERSİN": 33, "İSTANBUL": 34, "İZMİR": 35, "KARS": 36, "KASTAMONU": 37, "KAYSERİ": 38, "KIRKLARELİ": 39, "KIRŞEHİR": 40,
    "KOCAELİ": 41, "KONYA": 42, "KÜTAHYA": 43, "MALATYA": 44, "MANİSA": 45, "KAHRAMANMARAŞ": 46, "MARDİN": 47, "MUĞLA": 48,
    "MUŞ": 49, "NEVŞEHİR": 50, "NİĞDE": 51, "ORDU": 52, "RİZE": 53, "SAKARYA": 54, "SAMSUN": 55, "SİİRT": 56,
    "SİNOP": 57, "SİVAS": 58, "TEKİRDAĞ": 59, "TOKAT": 60, "TRABZON": 61, "TUNCELİ": 62, "ŞANLIURFA": 63, "UŞAK": 64,
    "VAN": 65, "YOZGAT": 66, "ZONGULDAK": 67, "AKSARAY": 68, "BAYBURT": 69, "KARAMAN": 70, "KIRIKKALE": 71, "BATMAN": 72,
    "ŞIRNAK": 73, "BARTIN": 74, "ARDAHAN": 75, "IĞDIR": 76, "YALOVA": 77, "KARABÜK": 78, "KİLİS": 79, "OSMANİYE": 80, "DÜZCE": 81
}

def clean_and_convert(val):
    if pd.isna(val) or val == "":
        return 0
    try:
        val_str = str(val).split(',')[0].split('.')[0].replace('.', '')
        return int(val_str)
    except:
        return 0

try:
    print("Abone dosyası okunuyor...")
    df = pd.read_excel(file_path)

    # 1. il sütununda "İl Adı" satırlarını sil
    if 'il' in df.columns:
        df = df[df['il'] != "İl Adı"]

    # 2. İl isimlerindeki özel düzeltmeler
    df['il'] = df['il'].replace({
        "K.MARAŞ": "KAHRAMANMARAŞ",
        "HAKKÂRİ": "HAKKARİ",
        "AFYONKARAHİSAR.": "AFYONKARAHİSAR"
    }, regex=False)
    
    df['il'] = df['il'].astype(str).str.upper().str.replace('İ', 'İ').str.replace('I', 'I').str.strip()

    # 3. Sayısal sütunlarda temizlik
    for col in ['Aydinlatma', 'Mesken', 'Sanayi', 'Tarim', 'Ticarethane']:
        if col in df.columns:
            df[col] = df[col].apply(clean_and_convert)

    # 4. Ay isimlerini numaralara çevir
    if 'Ay' in df.columns:
        df['Ay'] = df['Ay'].astype(str).str.lower().str.strip().map(MONTH_MAP).fillna(df['Ay'])

    # 5. Sütun adlarını İngilizce yap
    ingilizce_basliklar = {
        'il': 'Province',
        'Yil': 'Year',
        'Ay': 'Month',
        'Aydinlatma': 'Lighting',
        'Ticarethane': 'Public',
        'Mesken': 'Residential',
        'Sanayi': 'Industrial',
        'Tarim': 'Agricultural'
    }
    df = df.rename(columns=ingilizce_basliklar)

    # --- EKLEME: Tüketim Dosyasını Birleştirme ---
    print("Tüketim dosyası okunuyor...")
    df_cons = pd.read_excel(consumption_path)
    df_cons.columns = [c.strip() for c in df_cons.columns]
    
    if 'Province' in df_cons.columns:
        df_cons['Province'] = df_cons['Province'].astype(str).str.upper().str.replace('İ', 'İ').str.replace('I', 'I').str.strip()

    # Birleştirme
    df = pd.merge(df, df_cons[['Province', 'Year', 'Month', 'Consumption']], 
                  on=['Province', 'Year', 'Month'], 
                  how='left')

    # Consumption temizliği
    if 'Consumption' in df.columns:
        df['Consumption'] = df['Consumption'].apply(clean_and_convert)

    # 6. Plaka (Plate) sütununu oluşturma
    df['Plate'] = df['Province'].map(plaka_kodlari)

    # 7. Temizlik
    df = df.dropna(subset=['Province', 'Month', 'Plate'])
    df['Plate'] = df['Plate'].astype(int)

    # 8. Sıralama
    df = df.sort_values(by=['Plate', 'Year', 'Month'], ascending=True)

    # 9. Sütun dizilimini sabitle (Total_Subscribers kaldırıldı)
    yeni_sira = [
        'Plate', 'Province', 'Year', 'Month', 'Consumption', 
        'Lighting', 'Public', 'Residential', 'Industrial', 'Agricultural'
    ]
    
    df = df[[c for c in yeni_sira if c in df.columns]]

    # 10. Kaydet
    df.to_excel(output_path, index=False)

    print("-" * 30)
    print(f"İŞLEM BAŞARIYLA TAMAMLANDI!")
    print(f"Abone ve Tüketim verileri birleştirildi. (Toplam Abone sütunu çıkarıldı)")
    print(f"Çıktı Dosyası: {output_path}")

except Exception as e:
    print(f"Bir hata oluştu: {e}")

Abone dosyası okunuyor...
Tüketim dosyası okunuyor...
------------------------------
İŞLEM BAŞARIYLA TAMAMLANDI!
Abone ve Tüketim verileri birleştirildi. (Toplam Abone sütunu çıkarıldı)
Çıktı Dosyası: ..\..\processed_data\steps\03_2020_2021_final.xlsx
